# Fed-batch bioprocess simulator: quick start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ftwigg/bioprocess_sim/blob/pid_sims_fixes/notebooks/fedbatch_simulator_colab.ipynb)

One fed-batch run from start to finish, with plots, in about a minute. It works in Google Colab (the first cell clones the repository) and in a local Jupyter session started from a clone.

The run is an *E. coli*-like culture on glucose in a 5 L lab reactor: a batch phase, an exponential feed that is then held constant, and a feed stop at the end.

For all three worked examples see `fedbatch_simulator_examples.ipynb`.

## 1. Setup

In [ ]:
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/ftwigg/bioprocess_sim.git"
BRANCH = "pid_sims_fixes"   # change to "main" once this branch has been merged


def locate_repo():
    """Use the checkout this notebook lives in, or clone one next to it (Colab)."""
    here = pathlib.Path.cwd()
    for folder in [here, *here.parents]:
        if (folder / "examples" / "_bootstrap.py").exists():
            return folder
    target = here / "bioprocess_sim"
    if target.exists() and not (target / "examples" / "_bootstrap.py").exists():
        raise RuntimeError(
            f"{target} exists but is not the {BRANCH} branch. "
            "Delete that folder and run this cell again.")
    if not target.exists():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", BRANCH,
                        REPO_URL, str(target)], check=True)
    return target


REPO = locate_repo()
sys.path.insert(0, str(REPO / "examples"))
import _bootstrap  # noqa: F401  registers the repository as the package `fedbatch_simulator`

import fedbatch_simulator
print(f"Simulator {fedbatch_simulator.__version__} loaded from {REPO}")

In [ ]:
import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt

from fedbatch_simulator import *
from fedbatch_simulator.kinetics import CellParameters
from fedbatch_simulator.mass_transfer import DynamicKLa
from fedbatch_simulator.thermodynamics import TemperatureController
from fedbatch_simulator.feed_strategies import ConstantFeed, ExponentialFeed, DOStatFeed, PiecewiseFeed
from fedbatch_simulator.control import PICascade, SimplifiedCascade, FixedAerationAgitation
from fedbatch_simulator.reactor import FedBatchReactor
from fedbatch_simulator.simulator import FedBatchSimulator
from fedbatch_simulator.plotting import plot_fedbatch_results, plot_comparison, do_saturation

DO_SAT = do_saturation(37.0)   # mmol/L in equilibrium with air at 37°C
print(f"Python {sys.version.split()[0]}, NumPy {np.__version__}, SciPy {scipy.__version__}, "
      f"Matplotlib {matplotlib.__version__}")
print(f"DO at air saturation: {DO_SAT:.4f} mmol/L")

## 2. Define the run

Everything is wrapped in one function so that the experiments further down are one-liners. The arguments are the knobs worth turning first.

In [ ]:
def run_fedbatch(mu_set=0.12, glucose_cap=4.5, batch_glucose=10.0,
                 t_feed_start=5.5, t_feed_stop=22.0, t_end=26.0,
                 DO_setpoint=0.06, do_controller=PICascade):
    """
    Batch, then exponential feed capped at `glucose_cap`, then feed off.

    mu_set:        growth rate the exponential feed aims for (1/h)
    glucose_cap:   highest glucose feed rate (g/h); the oxygen supply covers about 4.5
    batch_glucose: glucose in the initial medium (g/L)
    t_feed_start:  feed start (h); 10 g/L of batch glucose runs out at about 5.3 h
    t_feed_stop:   feed stop (h)
    DO_setpoint:   dissolved oxygen setpoint (mmol/L); 0.06 is about 27% of air saturation
    do_controller: PICascade (real dynamics) or SimplifiedCascade (idealised)
    """
    # Cells: E. coli-like defaults on glucose
    carbon = SubstrateParameters(name='carbon', Ks=0.1, Y_xs=0.5, ms=0.03)
    nitrogen = SubstrateParameters(name='nitrogen', Ks=0.05, Y_xs=10.0, ms=0.001)
    cell_params = CellParameters(
        mu_max=0.5, carbon_source=GLUCOSE, biomass_composition=STANDARD_BIOMASS,
        substrates={'carbon': carbon, 'nitrogen': nitrogen}, Y_xs=0.5)

    # Reactor and starting point
    reactor_config = LAB_STR_5L
    initial_state = ReactorState(X=0.5, S_carbon=batch_glucose, S_nitrogen=2.0,
                                 DO=0.2, DCO2=0.5, V=2.0, T=37.0, N=300, pH=7.0)

    # Feed: sized from the biomass the batch leaves behind, capped by oxygen
    feed = FeedComposition(S_carbon=500.0, S_nitrogen=50.0, temperature=25.0)
    X_batch_end = initial_state.X + 0.48 * batch_glucose
    F0 = ExponentialFeed.initial_rate(mu_set=mu_set, X=X_batch_end, V=initial_state.V,
                                      S_feed=feed.S_carbon, Y_xs=carbon.Y_xs, ms=carbon.ms)
    exp_feed = ExponentialFeed(feed, F0=F0, mu_set=mu_set, F_max=glucose_cap / feed.S_carbon)
    exp_feed.t_start = t_feed_start
    feed_strategy = PiecewiseFeed([(0.0, ConstantFeed(feed, 0.0)),
                                   (t_feed_start, exp_feed),
                                   (t_feed_stop, ConstantFeed(feed, 0.0))])

    # Controllers
    do_control = do_controller(
        DO_setpoint=DO_setpoint, N_min=reactor_config.N_min, N_max=reactor_config.N_max,
        Q_gas_min=reactor_config.Q_gas_min * initial_state.V * 60,
        Q_gas_max=reactor_config.Q_gas_max * initial_state.V * 60,
        kLa_correlation=DynamicKLa(k_O2=reactor_config.k_O2, a=reactor_config.a,
                                   b=reactor_config.b, k_X=reactor_config.k_X))
    temp_controller = TemperatureController(T_setpoint=37.0, K_p=10.0, K_i=0.5)

    reactor = FedBatchReactor(cell_params=cell_params, reactor_config=reactor_config,
                              initial_state=initial_state, feed_strategy=feed_strategy,
                              do_control=do_control, temp_controller=temp_controller)
    results = FedBatchSimulator(reactor).simulate(t_end=t_end, max_step=0.1)
    report = check_balances(results, cell_params, feed_strategy)
    return results, report

## 3. Run it

In [ ]:
results, report = run_fedbatch()
print()
print(format_balances(report))

## 4. Look at it

What to look for:

- **Dissolved oxygen** spikes when the batch glucose runs out (about 5.3 h) and again when the feed stops (22 h). In between, the controller holds it at the setpoint.
- **Carbon substrate** sits at a few tens of mg/L while feeding: the culture is carbon-limited, eating glucose as fast as it arrives.
- **Agitation** saturates first, then **gas flow** takes over. That is the cascade.
- **Respiratory quotient** is just above 1 while growing on glucose and about 0.95 once starved, when the cells are oxidising their own biomass.
- **Specific growth rate** goes slightly negative after the feed stops: slow endogenous decay, not a flat line.

In [ ]:
fig = plot_fedbatch_results(results, "Fed-batch on glucose: batch, exponential feed, feed stop",
                            DO_setpoint=0.06, marks={'feed on': 5.5, 'feed off': 22.0})
plt.show()

In [ ]:
t = results.time
DO_pct = results.DO / DO_SAT * 100
feeding = (t > 6.0) & (t < 22.0)
i_peak = np.argmax(results.X * results.V)

print(f"Peak biomass:                    {results.X[i_peak] * results.V[i_peak]:.1f} g "
      f"({results.X[i_peak]:.1f} g/L) at {t[i_peak]:.1f} h")
print(f"Volumetric productivity to peak: {results.X[i_peak] / t[i_peak]:.2f} g/L/h")
print(f"DO spike at glucose exhaustion:  {DO_pct[(t > 4.5) & (t < 5.8)].max():.0f}% of saturation")
print(f"DO spike at feed stop:           {DO_pct[t > 22.0].max():.0f}% of saturation")
print(f"DO while feeding:                {DO_pct[feeding].min():.0f}-{DO_pct[feeding].max():.0f}% of saturation")
print(f"Glucose while feeding:           at most {results.S_carbon[feeding].max() * 1000:.0f} mg/L")
print(f"Peak oxygen uptake rate:         {results.OUR.max():.0f} mmol/L/h")

## 5. Experiment: feed faster than the oxygen supply can burn

The 4.5 g/h cap keeps this reactor inside its oxygen-transfer capacity for the whole run. Raise the cap to 9 g/h and the culture runs out of oxygen part-way through: agitation and gas flow hit their limits, dissolved oxygen falls to zero, oxygen uptake is pinned to what can be transferred, and the glucose that cannot be burned accumulates.

Biomass still ends higher, because oxygen-limited growth is faster than the capped feed allowed. But the run is no longer carbon-limited, and a real culture would respond to the excess glucose with overflow metabolism (acetate), which this model does not include.

In [ ]:
results_over, report_over = run_fedbatch(glucose_cap=9.0)

fig = plot_comparison({'cap 4.5 g/h': results, 'cap 9 g/h': results_over},
                      "Feeding within and beyond the oxygen-transfer ceiling")
plt.show()

t_over = results_over.time
DO_over = results_over.DO / DO_SAT * 100
fed = (t_over > 6.0) & (t_over < 22.0)
t_limited = t_over[fed][np.argmax(DO_over[fed] < 5.0)]

print(f"Cap 9 g/h: DO falls below 5% of saturation at {t_limited:.1f} h")
print(f"           glucose builds up to {results_over.S_carbon[fed].max():.1f} g/L by the time the feed stops")
print(f"           peak biomass {results_over.X.max():.1f} g/L, against {results.X.max():.1f} g/L with the 4.5 g/h cap")
print(f"Carbon accounted for: {report_over['carbon_closure_pct']:.2f}%, "
      f"lowest glucose value {report_over['min_S_carbon']:.1e} g/L")

## 6. More things to try

Each is one call to `run_fedbatch` followed by `plot_fedbatch_results` or `plot_comparison`:

- `run_fedbatch(do_controller=SimplifiedCascade)`: the idealised controller. DO sits on the setpoint with no spikes.
- `run_fedbatch(mu_set=0.2)`: a faster exponential phase that reaches the feed cap sooner.
- `run_fedbatch(t_feed_start=7.0)`: a longer gap between batch end and feed start, and a longer starvation dip.
- `run_fedbatch(DO_setpoint=0.10)`: a higher setpoint leaves less driving force, so the ceiling binds sooner.

In [ ]:
results_ideal, _ = run_fedbatch(do_controller=SimplifiedCascade)

fig = plot_comparison({'PI cascade': results, 'Idealised cascade': results_ideal},
                      "Real controller dynamics against the idealised cascade")
plt.show()